# <font color="#418FDE" size="6.5" uppercase>**Expansion Binning**</font>

>Last update: 20260713.
    
By the end of this Lecture, you will be able to:
- Use binarization and discretization to convert continuous features into thresholded or binned representations. 
- Generate polynomial, interaction, and spline features for nonlinear modeling. 
- Build leakage-safe preprocessing steps using transformers inside pipelines. 


## **1. Thresholds and Bins**

### **1.1. Threshold Based Binarization**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_04/Lecture_B/image_01_01.jpg?v=1783951426" width="250">



>* Convert numbers into yes-or-no indicators
>* Use meaningful cutoffs for clearer interpretation

>* Highlights meaningful cutoff-based changes
>* Loses detail from original values

>* Choose thresholds carefully from valid sources
>* Avoid leakage and evaluate fairness impacts



In [ ]:
#@title Python Code - Threshold Based Binarization

# Demonstrate threshold based binarization.
# Convert continuous values into indicators.
# Keep preprocessing simple and transparent.

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Create small continuous temperature readings.
temperatures_c = np.array([18, 21, 24, 27, 30, 33, 36, 39])
threshold_c = 30
above_threshold = (temperatures_c > threshold_c).astype(int)

# Store original and binary features together.
data = pd.DataFrame({
    "temperature_c": temperatures_c,
    "above_30_c": above_threshold,
})

# Validate the transformed feature length.
assert len(data) == len(temperatures_c)
assert set(data["above_30_c"]).issubset({0, 1})

# Print a compact teaching summary.
print("Threshold based binarization example")
print(f"Threshold: temperature > {threshold_c} Celsius")
print(data.to_string(index=False))
print("Values above threshold become 1; others become 0.")

# Plot original values and threshold line.
plt.figure(figsize=(7, 4))
plt.scatter(range(len(temperatures_c)), temperatures_c, c=above_threshold, s=90)
plt.axhline(threshold_c, color="red", linestyle="--", label="threshold")
plt.xlabel("Observation number")
plt.ylabel("Temperature in Celsius")
plt.title("Threshold Based Binarization")
plt.legend()
plt.tight_layout()
plt.show()



### **1.2. Discretizing Continuous Features**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_04/Lecture_B/image_01_02.jpg?v=1783951428" width="250">



>* Convert continuous values into meaningful bins
>* Capture nonlinear patterns across value ranges

>* Reduces noise and ignores tiny differences
>* Captures threshold-based changes across value ranges

>* Bins trade detail for simpler patterns
>* Choose boundaries carefully and evaluate impact



In [ ]:
#@title Python Code - Discretizing Continuous Features

# Discretization turns measurements into useful categories.
# Bins can reveal step like patterns.
# This example uses safe manual preprocessing.

import numpy as np
import pandas as pd

# Create a tiny continuous temperature feature.
temperatures_c = np.array([8, 12, 17, 21, 26, 31, 36, 41])

# Define meaningful bin edges in Celsius.
bin_edges = [-np.inf, 15, 25, 35, np.inf]
bin_labels = ["cold", "mild", "warm", "hot"]

# Convert each temperature into one bin label.
temperature_bins = pd.cut(
    temperatures_c,
    bins=bin_edges,
    labels=bin_labels,
)

# Create one hot columns from bin labels.
bin_columns = pd.get_dummies(temperature_bins, prefix="temp")

# Build a compact teaching table.
result = pd.DataFrame({"temperature_c": temperatures_c})
result = pd.concat([result, bin_columns], axis=1)

# Validate the transformed table size.
expected_columns = 1 + len(bin_labels)
assert result.shape == (len(temperatures_c), expected_columns)

# Print only a small readable summary.
print("Original values become interval based features:")
print(result.to_string(index=False))
print("Each row activates exactly one temperature bin.")



### **1.3. Choosing Bin Strategies**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_04/Lecture_B/image_01_03.jpg?v=1783951430" width="250">



>* Equal-width bins preserve consistent numeric intervals
>* Skewed data can create unhelpful sparse bins

>* Quantile bins balance observations across groups
>* They may reduce original-scale interpretability

>* Use meaningful, domain-informed bin cutoffs
>* Balance bin count for stable interpretation



In [ ]:
#@title Python Code - Choosing Bin Strategies

# Compare binning strategies for skewed measurements.
# Equal width preserves original unit distances.
# Quantile bins balance observation counts better.

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Create a small skewed income example.
rng = np.random.default_rng(seed=7)
incomes = rng.lognormal(mean=10.6, sigma=0.75, size=80)

# Add a few high income outliers.
outliers = np.array([180000, 240000, 320000, 500000])
incomes = np.concatenate([incomes, outliers])

# Validate the one dimensional feature.
assert incomes.ndim == 1 and incomes.size > 10
income_series = pd.Series(incomes, name="income_usd")

# Build equal width and quantile bins.
equal_bins = pd.cut(income_series, bins=4)
quantile_bins = pd.qcut(income_series, q=4, duplicates="drop")

# Count observations in each strategy.
equal_counts = equal_bins.value_counts(sort=False)
quantile_counts = quantile_bins.value_counts(sort=False)

# Print compact summaries for learners.
print("Equal-width bin counts:", equal_counts.to_list())
print("Quantile bin counts:", quantile_counts.to_list())
print("Income range: $%d to $%d" % (incomes.min(), incomes.max()))

# Plot both count patterns side by side.
fig, axes = plt.subplots(1, 2, figsize=(9, 3))
axes[0].bar(range(len(equal_counts)), equal_counts.values)
axes[0].set_title("Equal-width bins")

# Show quantile bins with balanced counts.
axes[1].bar(range(len(quantile_counts)), quantile_counts.values)
axes[1].set_title("Quantile bins")
axes[1].set_xlabel("Bin number")

# Label the shared vertical meaning.
axes[0].set_ylabel("Number of observations")
axes[0].set_xlabel("Bin number")
plt.tight_layout()
plt.show()



## **2. Feature Expansion**

### **2.1. Polynomial Feature Expansion**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_04/Lecture_B/image_02_01.jpg?v=1783951421" width="250">



>* Adds powers to model curved patterns
>* Keeps originals while expanding model flexibility

>* Higher powers capture curved feature effects
>* Use scaling and regularization for stability

>* Choose polynomial degree carefully
>* Validate with leakage-safe preprocessing



In [ ]:
#@title Python Code - Polynomial Feature Expansion

# Demonstrate polynomial feature expansion clearly.
# Use tiny data for fast execution.
# Compare original and expanded columns.

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Create one continuous feature in meters.
meters = np.array([1, 2, 3, 4, 5], dtype=float)

# Validate the feature shape before expansion.
assert meters.ndim == 1 and meters.size == 5

# Build polynomial columns manually without sklearn.
expanded = pd.DataFrame({
    "meters": meters,
    "meters_squared": meters ** 2,
    "meters_cubed": meters ** 3,
})

# Show only the small expanded table.
print("Polynomial expansion adds curved building blocks:")
print(expanded.to_string(index=False))

# Create a smooth curve using expanded features.
x_grid = np.linspace(1, 5, 100)
y_curve = 2 + 0.8 * x_grid - 0.15 * x_grid ** 2

# Plot the nonlinear pattern from polynomial terms.
plt.figure(figsize=(6, 4))
plt.plot(x_grid, y_curve, color="navy", label="Polynomial curve")
plt.scatter(meters, 2 + 0.8 * meters - 0.15 * meters ** 2, color="orange")
plt.xlabel("Distance in meters")
plt.ylabel("Example target value")
plt.title("Polynomial Features Can Represent Curvature")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()



### **2.2. Interaction Features**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_04/Lecture_B/image_02_02.jpg?v=1783951419" width="250">



>* Combine features to capture joint effects
>* Model context-dependent, non-additive relationships

>* Combine features to reveal predictive joint patterns
>* Use encoded interactions for context-specific effects

>* Use interactions carefully to avoid overfitting
>* Build them safely inside preprocessing pipelines



In [ ]:
#@title Python Code - Interaction Features

# Interaction features reveal joint feature effects.
# This example uses only built-in Python.
# We multiply features to model combined influence.

# Store tiny building examples as dictionaries.
buildings = [
    {"temp_c": 22, "people": 3, "energy": 58},
    {"temp_c": 30, "people": 3, "energy": 76},

    {"temp_c": 22, "people": 8, "energy": 72},
    {"temp_c": 30, "people": 8, "energy": 112},
]

# Validate the tiny dataset before processing.
assert len(buildings) == 4, "Expected four teaching rows."
assert all("temp_c" in row for row in buildings), "Missing temperature."

# Create an interaction by multiplying two features.
for row in buildings:
    row["temp_people"] = row["temp_c"] * row["people"]

# Print a compact table header.
print("temp_c people interaction energy")

# Show each row without printing large data.
for row in buildings:
    print(row["temp_c"], row["people"], row["temp_people"], row["energy"])

# Compare separate changes with joint changes.
low_people_gain = buildings[1]["energy"] - buildings[0]["energy"]
high_people_gain = buildings[3]["energy"] - buildings[2]["energy"]

# Explain why the interaction is useful.
print("Temperature gain with 3 people:", low_people_gain)
print("Temperature gain with 8 people:", high_people_gain)
print("Different gains suggest an interaction feature helps.")



### **2.3. Spline Feature Basis**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_04/Lecture_B/image_02_03.jpg?v=1783951423" width="250">



>* Smooth local features model changing numeric effects
>* More flexible than lines, steadier than polynomials

>* Knots guide localized spline feature behavior
>* Splines create smoother transitions than binning

>* Linear models learn flexible nonlinear curves
>* Choose knots carefully and avoid leakage



In [ ]:
#@title Python Code - Spline Feature Basis

# Build spline features without external transformers.
# Compare smooth basis functions visually.
# Keep preprocessing learned from training data.

import numpy as np
import matplotlib.pyplot as plt

# Create a small deterministic temperature dataset.
rng = np.random.default_rng(7)
x_train = np.linspace(0.0, 40.0, 25)

# Simulate nonlinear electricity demand values.
y_train = 30 + 0.04 * (x_train - 20) ** 2
y_train = y_train + rng.normal(0.0, 1.2, x_train.size)

# Learn knot locations from training data only.
knots = np.quantile(x_train, [0.25, 0.50, 0.75])
assert knots.size == 3 and x_train.size == y_train.size

# Define a simple cubic truncated spline basis.
def spline_basis(values, learned_knots):
    values = np.asarray(values, dtype=float)
    columns = [np.ones_like(values), values]
    for knot in learned_knots:
        columns.append(np.maximum(values - knot, 0.0) ** 3)
    return np.column_stack(columns)

# Fit linear weights on expanded spline features.
X_spline = spline_basis(x_train, knots)
weights = np.linalg.lstsq(X_spline, y_train, rcond=None)[0]

# Predict smoothly across the observed temperature range.
x_grid = np.linspace(x_train.min(), x_train.max(), 200)
y_grid = spline_basis(x_grid, knots) @ weights

# Print a compact summary for learners.
print(f"Training rows: {x_train.size}, spline features: {X_spline.shape[1]}")
print("Knots learned from training temperatures:", np.round(knots, 1))
print("Each knot adds one localized nonlinear feature.")

# Plot data, knots, and the learned spline curve.
plt.figure(figsize=(7, 4))
plt.scatter(x_train, y_train, label="training data", color="black")
plt.plot(x_grid, y_grid, label="linear model on spline basis")
for knot in knots:
    plt.axvline(knot, color="gray", linestyle="--", alpha=0.6)
plt.xlabel("Outdoor temperature in degrees Celsius")
plt.ylabel("Electricity demand in kWh")
plt.title("Spline Feature Basis for Nonlinear Modeling")
plt.legend()
plt.tight_layout()
plt.show()



## **3. Safe Pipeline Transformations**

### **3.1. Custom Function Transformers**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_04/Lecture_B/image_03_01.jpg?v=1783951412" width="250">



>* Add domain rules as pipeline steps
>* Make transformations repeatable and deployable

>* Keep custom transformations predictable and consistent
>* Use pipelines to avoid manual processing errors

>* Use only record or training information
>* Pipelines prevent validation and test leakage



### **3.2. Reversing Transformations**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_04/Lecture_B/image_03_02.jpg?v=1783951415" width="250">



>* Map transformed features back to understandable values
>* Use pipeline transformers for consistent reversal

>* Reverse transformations for interpretation and quality checks
>* Some transformations lose original value details

>* Fit transformation parameters on training data only
>* Keep inverse steps inside auditable pipelines



In [ ]:
#@title Python Code - Reversing Transformations

# Demonstrate reversible preprocessing without external machine learning.
# Fit parameters only on training values.
# Restore transformed values for interpretation checks.

import numpy as np
import pandas as pd

# Create tiny income data in dollars.
income = np.array([42000, 51000, 58000, 76000, 99000, 125000])
train_income = income[:4]
test_income = income[4:]

# Learn scaling parameters from training only.
train_mean = train_income.mean()
train_std = train_income.std(ddof=0)
assert train_std > 0

# Transform both splits using training parameters.
train_scaled = (train_income - train_mean) / train_std
test_scaled = (test_income - train_mean) / train_std
assert train_scaled.shape == train_income.shape

# Reverse transformed values using same parameters.
restored_test = (test_scaled * train_std) + train_mean
max_error = np.abs(restored_test - test_income).max()

# Show compact results for learners.
summary = pd.DataFrame({"original": test_income, "scaled": test_scaled.round(2), "restored": restored_test.round(0).astype(int)})
print("Training mean and std learned safely:", round(train_mean, 1), round(train_std, 1))
print(summary.to_string(index=False))
print("Maximum restoration error:", round(float(max_error), 10))



### **3.3. Prevent Data Leakage**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_04/Lecture_B/image_03_03.jpg?v=1783951417" width="250">



>* Pipelines stop test data influencing preprocessing
>* Fit transformers only on training data

>* Learned transformations can leak unseen data patterns
>* Pipelines keep validation folds truly unseen

>* Fit learned preprocessing on training data only
>* Pipelines give honest, deployment-ready evaluation scores



In [ ]:
#@title Python Code - Prevent Data Leakage

# Demonstrate leakage using simple preprocessing.
# Compare manual fitting with safe pipelines.
# Keep outputs short and beginner friendly.

import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import KBinsDiscretizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score

# Create deterministic synthetic income data.
rng = np.random.default_rng(42)
income = rng.normal(50000, 12000, 120)

# Add shifted test-like high incomes.
income[-30:] = rng.normal(85000, 8000, 30)
readmit = (income > 62000).astype(int)

# Reshape features for transformers.
X = income.reshape(-1, 1)
y = readmit.copy()

# Split before any learned preprocessing.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, shuffle=False
)

# Validate the tiny example sizes.
assert X_train.shape[0] > 10 and X_test.shape[0] > 5

# Leaky approach fits bins on all data.
leaky_binner = KBinsDiscretizer(
    n_bins=4, encode="onehot-dense", strategy="quantile"
)

# Transform train and test after leakage.
X_all_binned = leaky_binner.fit_transform(X)
X_train_leaky = X_all_binned[: len(X_train)]
X_test_leaky = X_all_binned[len(X_train) :]

# Train a small silent classifier.
leaky_model = LogisticRegression(max_iter=200)
leaky_model.fit(X_train_leaky, y_train)
leaky_score = accuracy_score(y_test, leaky_model.predict(X_test_leaky))

# Safe pipeline fits bins only on training data.
safe_pipe = Pipeline(
    [("bins", KBinsDiscretizer(n_bins=4, encode="onehot-dense", strategy="quantile")),
     ("model", LogisticRegression(max_iter=200))]
)

# Fit pipeline using training data only.
safe_pipe.fit(X_train, y_train)
safe_score = accuracy_score(y_test, safe_pipe.predict(X_test))

# Show learned bin edges for comparison.
leaky_edges = np.round(leaky_binner.bin_edges_[0], 0)
safe_edges = np.round(safe_pipe.named_steps["bins"].bin_edges_[0], 0)

# Print concise teaching results.
print("Leaky bin edges:", leaky_edges.tolist())
print("Safe bin edges:", safe_edges.tolist())
print("Leaky accuracy:", round(leaky_score, 3))
print("Safe pipeline accuracy:", round(safe_score, 3))
print("Lesson: fit preprocessing inside the pipeline.")



# <font color="#418FDE" size="6.5" uppercase>**Expansion Binning**</font>


In this lecture, you learned to:
- Use binarization and discretization to convert continuous features into thresholded or binned representations. 
- Generate polynomial, interaction, and spline features for nonlinear modeling. 
- Build leakage-safe preprocessing steps using transformers inside pipelines. 

In the next Module (Module 5), we will go over 'Categoricals Missingness'